# Smart Agri: so sánh ResNet50 và ViT-B/16 trên Google Colab

Notebook này fine-tune **ResNet50** và **ViT-B/16** pretrained trên ImageNet, dùng cùng ảnh, cùng cách chia nhóm ảnh nguồn, cùng tiền xử lý và cùng chỉ số. Chọn checkpoint theo **Macro-F1 trên validation**; tập test chỉ dùng để đánh giá cuối.

> Bật GPU trước khi chạy: **Runtime → Change runtime type → T4 GPU** (hoặc GPU khả dụng). Đây là thí nghiệm so sánh backbone, không tự thay model đang chạy trong app.

Baseline báo cáo cũ của ResNet50+SVM (Accuracy 95,93%, Macro-F1 88,98%) là số liệu tham khảo lịch sử. Notebook huấn luyện lại ResNet50 và ViT trên cùng split theo ảnh nguồn để tạo phép so sánh công bằng; không so trực tiếp hai quy trình train khác nhau.

In [ ]:
%pip -q install scikit-learn pandas pillow matplotlib seaborn

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json, re, random, time, gc
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torchvision.models import ResNet50_Weights, ViT_B_16_Weights
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from google.colab import drive

SEED = 42
EPOCHS = 10
BATCH_SIZE = 16  # T4-compatible starting point; lower to 8 if GPU memory is insufficient.
PATIENCE = 3
NUM_WORKERS = 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Switch the Colab runtime to a GPU before training.')

drive.mount('/content/drive')
DATA_ZIP = Path('/content/drive/MyDrive/smart_agri_colab_data.zip')  # Upload the prepared zip here.
WORK_ROOT = Path('/content/smart_agri_colab')
OUTPUT_DIR = Path('/content/drive/MyDrive/smart_agri_colab_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_ZIP.is_file(), f'Upload smart_agri_colab_data.zip to Google Drive: {DATA_ZIP}'

WORK_ROOT.mkdir(parents=True, exist_ok=True)
with ZipFile(DATA_ZIP) as archive:
    root_resolved = WORK_ROOT.resolve()
    for member in archive.infolist():
        target = (WORK_ROOT / member.filename).resolve()
        if target != root_resolved and root_resolved not in target.parents:
            raise ValueError(f'Unsafe archive path: {member.filename}')
    archive.extractall(WORK_ROOT)

DATA_ROOT = WORK_ROOT / 'classification_dataset'
assert (DATA_ROOT / 'train').is_dir() and (DATA_ROOT / 'val').is_dir()
bundle = json.loads((WORK_ROOT / 'colab_bundle_manifest.json').read_text(encoding='utf-8'))
class_names = sorted(bundle['classes'])
print('Images:', bundle['image_count'], '| classes:', len(class_names))
print(pd.DataFrame(bundle['class_counts']).to_string())

## 1. Tạo một split cố định, không để ảnh crop cùng nguồn lọt qua các tập

Giữ nguyên logic `source_group` và `StratifiedGroupKFold` của dự án. Cả hai backbone sẽ dùng đúng các chỉ số train/validation/test này.

In [ ]:
all_samples = []
for split_name in ('train', 'val'):
    folder = ImageFolder(DATA_ROOT / split_name)
    if folder.classes != class_names:
        raise ValueError(f'Class order mismatch in {split_name}: {folder.classes}')
    all_samples.extend((path, target) for path, target in folder.samples)

paths = np.asarray([sample[0] for sample in all_samples])
labels = np.asarray([class_names[sample[1]] for sample in all_samples])

def source_group(image_path):
    stem = Path(image_path).stem
    stem = re.sub(r'_crop_\d+$', '', stem)
    stem = re.sub(r'\.rf\.[0-9a-fA-F]+$', '', stem)
    stem = re.sub(r'^r6k_(?:train|test)_', '', stem, flags=re.IGNORECASE)
    return stem

groups = np.asarray([source_group(path) for path in paths])
dummy = np.zeros((len(labels), 1), dtype=np.uint8)
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_val_idx, test_idx = next(outer.split(dummy, labels, groups))
inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + 1)
train_rel, val_rel = next(inner.split(dummy[train_val_idx], labels[train_val_idx], groups[train_val_idx]))
train_idx = train_val_idx[train_rel]
val_idx = train_val_idx[val_rel]
train_groups, val_groups, test_groups = (set(groups[idx]) for idx in (train_idx, val_idx, test_idx))
assert not (train_groups & val_groups or train_groups & test_groups or val_groups & test_groups)
print(f'Train: {len(train_idx):,} | validation: {len(val_idx):,} | test: {len(test_idx):,}')
print('Source groups are disjoint across all three splits.')
print(pd.DataFrame({'train': pd.Series(labels[train_idx]).value_counts(),
                    'validation': pd.Series(labels[val_idx]).value_counts(),
                    'test': pd.Series(labels[test_idx]).value_counts()}).fillna(0).astype(int).sort_index())

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class RiceImages(Dataset):
    def __init__(self, indices, transform):
        self.indices = np.asarray(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item])
        with Image.open(paths[index]) as source:
            image = ImageOps.exif_transpose(source).convert('RGB')
        return self.transform(image), int(class_names.index(labels[index]))

train_loader = DataLoader(RiceImages(train_idx, train_transform), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(RiceImages(val_idx, eval_transform), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))
test_loader = DataLoader(RiceImages(test_idx, eval_transform), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))
class_counts = np.bincount([class_names.index(label) for label in labels[train_idx]], minlength=len(class_names))
class_weights = torch.tensor(len(train_idx) / (len(class_names) * class_counts),
                             dtype=torch.float32, device=device)
print('Class-balanced cross-entropy weights prepared.')

In [ ]:
def build_model(name):
    if name == 'ResNet50':
        model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, len(class_names))
    elif name == 'ViT-B/16':
        model = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        model.heads.head = nn.Linear(model.heads.head.in_features, len(class_names))
    else:
        raise ValueError(f'Unknown architecture: {name}')
    return model.to(device)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    total_loss, total = 0.0, 0
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    for images, targets in loader:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=(device.type == 'cuda')):
            logits = model(images)
            loss = criterion(logits, targets)
        total_loss += loss.item() * len(targets)
        total += len(targets)
        y_true.extend(targets.cpu().numpy())
        y_pred.extend(logits.argmax(1).cpu().numpy())
    return total_loss / total, np.asarray(y_true), np.asarray(y_pred)

def train_model(name):
    model = build_model(name)
    head = model.fc if name == 'ResNet50' else model.heads.head
    head_params = list(head.parameters())
    head_ids = {id(parameter) for parameter in head_params}
    backbone_params = [parameter for parameter in model.parameters() if id(parameter) not in head_ids]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 2e-5},
        {'params': head_params, 'lr': 2e-4},
    ], weight_decay=0.01)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    amp_enabled = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    best_macro_f1, best_state, stale_epochs = -1.0, None, 0
    history = []
    checkpoint_path = OUTPUT_DIR / (name.lower().replace('/', '_').replace('-', '_') + '_best.pt')
    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, seen = 0.0, 0
        for images, targets in train_loader:
            images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
                logits = model(images)
                loss = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * len(targets)
            seen += len(targets)

        val_loss, y_val, pred_val = evaluate(model, val_loader)
        val_macro = f1_score(y_val, pred_val, average='macro', zero_division=0)
        history.append({'epoch': epoch, 'train_loss': running_loss / seen,
                        'val_loss': val_loss, 'val_macro_f1': val_macro})
        print(f'{name} | epoch {epoch}/{EPOCHS} | train loss {running_loss/seen:.4f} '
              f'| val loss {val_loss:.4f} | val macro-F1 {val_macro:.4f}')
        if val_macro > best_macro_f1:
            best_macro_f1, stale_epochs = val_macro, 0
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            torch.save({'architecture': name, 'class_names': class_names,
                        'state_dict': best_state, 'best_val_macro_f1': best_macro_f1,
                        'seed': SEED}, checkpoint_path)
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print('Early stopping: validation Macro-F1 did not improve.')
                break

    model.load_state_dict(best_state)
    elapsed = time.time() - start_time
    test_loss, y_test, pred_test = evaluate(model, test_loader)
    report = classification_report(y_test, pred_test, labels=np.arange(len(class_names)),
                                   target_names=class_names, output_dict=True, zero_division=0)
    summary = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred_test),
        'Macro-F1': f1_score(y_test, pred_test, average='macro', zero_division=0),
        'Weighted-F1': f1_score(y_test, pred_test, average='weighted', zero_division=0),
        'Test loss': test_loss,
        'Best validation Macro-F1': best_macro_f1,
        'Training seconds': elapsed,
        'Checkpoint MB': checkpoint_path.stat().st_size / (1024 ** 2),
    }
    pd.DataFrame(history).to_csv(OUTPUT_DIR / (name.replace('/', '_') + '_history.csv'), index=False)
    pd.DataFrame(report).transpose().to_csv(OUTPUT_DIR / (name.replace('/', '_') + '_per_class.csv'))
    del model, optimizer, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, report, y_test, pred_test

results = []
details = {}
for architecture in ('ResNet50', 'ViT-B/16'):
    summary, report, y_test, pred_test = train_model(architecture)
    results.append(summary)
    details[architecture] = (report, y_test, pred_test)
    pd.DataFrame(results).to_csv(OUTPUT_DIR / 'same_split_model_comparison.csv', index=False)

comparison = pd.DataFrame(results).set_index('Model')
display(comparison.round(4))
print('Results and best checkpoints saved in:', OUTPUT_DIR)

## 3. So sánh lỗi theo lớp

Chỉ kết luận ViT tốt hơn nếu Macro-F1 và các lớp bệnh yếu được cải thiện trên cùng tập test. Accuracy đơn lẻ có thể bị các lớp đông như Brown Spot chi phối.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

per_class_f1 = pd.DataFrame({
    name: [details[name][0][class_name]['f1-score'] for class_name in class_names]
    for name in details
}, index=class_names)
display(per_class_f1.round(3))
ax = per_class_f1.plot(kind='bar', figsize=(14, 6), ylim=(0, 1), rot=45)
ax.set_ylabel('F1-score on the same held-out test set')
ax.set_title('Per-class comparison: ResNet50 vs ViT-B/16')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_class_f1_comparison.png', dpi=160)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for axis, name in zip(axes, details):
    _, y_test, pred_test = details[name]
    ConfusionMatrixDisplay.from_predictions(y_test, pred_test, labels=np.arange(len(class_names)),
        display_labels=class_names, xticks_rotation=90, normalize='true', cmap='Blues', ax=axis, colorbar=False)
    axis.set_title(name)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'normalized_confusion_matrices.png', dpi=160)
plt.show()
print('Before claiming a win: inspect Leaf Blast, Leaf Scald and Narrow Brown Leaf Spot recall/F1, plus runtime and checkpoint size.')

## Hướng dẫn kết luận

- Dùng `same_split_model_comparison.csv` cho chỉ số tổng thể và các file `*_per_class.csv` cho từng bệnh.
- **Macro-F1** là chỉ số chính vì dữ liệu mất cân bằng; đối chiếu recall/F1 của từng lớp, đặc biệt các lớp ít ảnh.
- Test chỉ được dùng sau khi đã chọn checkpoint bằng validation Macro-F1. Không chỉnh tham số dựa trên test.
- Báo cáo ResNet50+SVM cũ (Accuracy 95,93%, Macro-F1 88,98%) là tham khảo, không phải đối chứng cùng quy trình. Trong báo cáo chính, so sánh ResNet50 fine-tuned mới với ViT-B/16 fine-tuned mới trên split cố định ở notebook.
- Mỗi mô hình chỉ được train một lượt ở seed 42 trong bản notebook này. Nếu thời gian cho phép, chạy lại với seed 21 và 84, rồi báo trung bình ± độ lệch chuẩn.
- File `*_best.pt` là checkpoint nghiên cứu, chưa tương thích trực tiếp với API hiện tại. Chỉ tích hợp sau khi hoàn tất đánh giá và viết adapter riêng.